# Notebook 10: App data

**Goal:** build the small, ready-to-use files the Streamlit app reads, so the app does no heavy computation at start-up.

**Inputs (in `notebooks/`):**
- `recipes_labeled.pkl` (05), `recipes_nutrients.pkl` (04), `interactions_all.pkl` (06), `catalog_ids.csv` (07)
- `ingredient_usda_map.csv`, `usda_nutrients_per100g.pkl`, `ingredient_portions.csv` (03), `default_units.csv` (04)
- `phase_rules.csv`, `phase_params.csv` (05, manual files)
- raw `../data/raw/recipes.csv` (instructions, ingredient text, images, keywords)

**Agreed decisions (14 Nov):**
1. The app recommends from the **catalog** (good + fair, not implausible: 12,375 recipes).
2. The app uses **all** interactions (train + test). The split was only for offline evaluation.
3. **Meal type** (default in the app: Main + Salads & sides) from a manual map `meal_type_map.csv` (RecipeCategory → type). Categories that say nothing about the meal (e.g. "< 60 Mins") are set to `AUTO` and resolved from the recipe's Keywords, then its name. Types: Main, Breakfast, Salads & sides, Snacks & appetizers, Desserts & baking, Sauces & dressings (hidden by default), Other.
4. **Pantry staples** (salt, pepper, water, oil, sugar, flour, ...) count as always available: manual file `pantry_staples.csv`.
5. **Own recipes** are scored with the **same** phase function as notebook 05. This notebook re-implements it and checks it reproduces `recipes_labeled.pkl` exactly, then saves the per-phase label thresholds.
6. Final Hybrid weights (notebook 09): Popularity 0, Content-B 0.25, Item-CF 0.25 (Jaccard k=10), User-CF 0.5 (Jaccard k=100), Phase 0.35.

**Outputs (`../app/data/`):**

| file | what |
|---|---|
| `recipes_app.parquet` | catalog recipes: name, meal type, time, ingredients, instructions, image, phase scores/labels/reasons, %DV per serving |
| `popularity.npy` | popularity rank (0–1) per catalog recipe |
| `tfidf_catalog.npz`, `tfidf_vocab.json` | Content-Based: TF-IDF rows of the catalog + vocabulary and IDF (for own recipes) |
| `itemknn_catalog.npz` | Item-CF: top-10 Jaccard neighbours between catalog recipes |
| `usercf_interactions.npz`, `usercf_liked.npz`, `usercf_counts.npy` | User-CF: users × catalog interactions, likes, and each user's total interactions |
| `ingredients_app.csv` | the ingredient list: USDA nutrients per 100 g + grams per cup/tbsp/tsp/piece/oz (own recipes) |
| `phase_rules.csv`, `phase_params.csv`, `phase_thresholds.csv`, `daily_values.csv` | phase scoring for own recipes |
| `pantry_staples.csv`, `onboarding_ids.csv`, `app_config.json` | staples, quick-start recipes, weights and settings |

## 0. Settings

In [1]:
import os
import re
import ast
import json
import html

import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.feature_extraction.text import TfidfVectorizer

pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 70)
pd.set_option("display.max_rows", 120)

# ---- inputs (path strings) ----
RECIPES_PKL = "recipes_labeled.pkl"
NUTRIENTS_PKL = "recipes_nutrients.pkl"
INTER_PKL = "interactions_all.pkl"
CATALOG_CSV = "catalog_ids.csv"
MAP_CSV = "ingredient_usda_map.csv"
PER100_PKL = "usda_nutrients_per100g.pkl"
PORTIONS_CSV = "ingredient_portions.csv"
DEFAULT_UNITS_CSV = "default_units.csv"
RULES_CSV = "phase_rules.csv"
PARAMS_CSV = "phase_params.csv"
RAW_RECIPES_CSV = "../data/raw/recipes.csv"

# ---- manual files (created once as drafts, never overwritten) ----
MEAL_MAP_CSV = "meal_type_map.csv"
STAPLES_CSV = "pantry_staples.csv"

# ---- output folder (path string) ----
OUT_DIR = "../app/data"

# ---- decisions ----
LIKE_THRESHOLD = 4
ITEM_K = 10
USER_K = 100
WEIGHTS = {"popularity": 0.0, "content": 0.25, "item_cf": 0.25, "user_cf": 0.5, "phase": 0.35, "pantry": 0.5}
MAX_MISSING_DEFAULT = 2          # "only recipes I can cook now": missing ingredients allowed
OWN_TOP10_CAP = 3                # own recipes: at most 3 in the top 10
OWN_UNKNOWN_MAX = 0.30           # own recipe gets no phase badge if > 30% of ingredients are unknown
CHUNK = 500

# phase scoring (must match notebook 05)
LABEL_QUALITY = ["good", "fair"]
MAX_PCT_DV = 200
MIN_KCAL_ELIGIBLE = 100
MAX_KCAL = 1200
DENSITY_CAP = 20
MIN_KCAL_DENSITY = 100
PHASES = ["menstrual", "follicular", "ovulatory", "luteal"]
DV = {  # nutrient -> (daily value, unit, display name)   (same as notebook 05)
    "iron_mg": (18, "mg", "Iron"), "vitc_mg": (90, "mg", "Vitamin C"), "magnesium_mg": (420, "mg", "Magnesium"),
    "calcium_mg": (1300, "mg", "Calcium"), "zinc_mg": (11, "mg", "Zinc"), "vitb6_mg": (1.7, "mg", "B6"),
    "folate_ug": (400, "mcg", "Folate"), "vitb12_ug": (2.4, "mcg", "B12"), "fiber_g": (28, "g", "Fiber"),
    "protein_g": (50, "g", "Protein"), "potassium_mg": (4700, "mg", "Potassium"),
}

for p in [RECIPES_PKL, NUTRIENTS_PKL, INTER_PKL, CATALOG_CSV, MAP_CSV, PER100_PKL, PORTIONS_CSV,
          DEFAULT_UNITS_CSV, RULES_CSV, PARAMS_CSV, RAW_RECIPES_CSV]:
    assert isinstance(p, str), f"{p!r} must be a path string"
    assert os.path.exists(p), f"{p} not found: fix the path"
os.makedirs(OUT_DIR, exist_ok=True)
out_path = lambda name: os.path.join(OUT_DIR, name)

## 1. Load data

In [2]:
recipes = pd.read_pickle(RECIPES_PKL).reset_index(drop=True)
recipes["RecipeId"] = recipes["RecipeId"].astype("int64")
recipes["Name"] = recipes["Name"].astype(str).map(html.unescape)
nut = pd.read_pickle(NUTRIENTS_PKL).reset_index(drop=True)
nut["RecipeId"] = nut["RecipeId"].astype("int64")
inter = pd.read_pickle(INTER_PKL)
inter["user_id"] = inter["user_id"].astype("int64")
inter["recipe_id"] = inter["recipe_id"].astype("int64")
catalog_ids = pd.read_csv(CATALOG_CSV)["recipe_id"].astype("int64").to_numpy()

assert (nut["RecipeId"].to_numpy() == recipes["RecipeId"].to_numpy()).all(), "recipes_nutrients and recipes_labeled are not in the same order"
n_cat = len(catalog_ids)
cat_index = pd.Series(np.arange(n_cat), index=catalog_ids)
rec_by_id = recipes.set_index("RecipeId")
print(f"recipes: {len(recipes):,} | catalog: {n_cat:,} | interactions: {len(inter):,} | users: {inter['user_id'].nunique():,}")

recipes: 42,736 | catalog: 12,375 | interactions: 164,235 | users: 60,183


## 2. Phase scoring function (re-implemented from notebook 05) + check

The app needs this function to score the user's own recipes. To be sure it is identical, we run it on all 42,736 recipes and compare with `recipes_labeled.pkl`: scores must match to rounding, labels exactly.

In [3]:
rules = pd.read_csv(RULES_CSV)
params = pd.read_csv(PARAMS_CSV).set_index("phase")
rules = rules[rules["nutrient"].isin(DV)].copy()


def phase_scores(pdv, kcal):
    """pdv: DataFrame of %DV per serving (columns = DV keys); kcal: Series of kcal per serving.
    Returns {phase: (score, gate, contrib, nutrient_list)} exactly as notebook 05."""
    dens = pdv.mul(100).div(kcal.clip(lower=MIN_KCAL_DENSITY), axis=0)
    out = {}
    for ph in PHASES:
        r = rules[rules["phase"] == ph]
        prm = params.loc[ph]
        nl = r["nutrient"].tolist()
        W = r["weight"].to_numpy(float)
        contrib = np.minimum(dens[nl].fillna(0).to_numpy(float), DENSITY_CAP) / DENSITY_CAP * W
        score = contrib.sum(axis=1) / W.sum()
        gate_cols = r.loc[r["gate"] == "Y", "nutrient"].tolist()
        gate = ((pdv[gate_cols] >= prm["gate_pct"]).sum(axis=1) >= prm["gate_min_count"]).to_numpy()
        out[ph] = (score, gate, contrib, nl)
    return out


def plausible_mask(pdv, kcal):
    return ((pdv.max(axis=1) <= MAX_PCT_DV) & (kcal <= MAX_KCAL) & (kcal >= MIN_KCAL_ELIGIBLE)).to_numpy()


pdv_all = pd.DataFrame({k: nut[f"{k}_ps"] / DV[k][0] * 100 for k in DV}, index=nut.index)
kcal_all = nut["Calories"]
res = phase_scores(pdv_all, kcal_all)
quality_ok = nut["quality"].isin(LABEL_QUALITY).to_numpy()
plaus = plausible_mask(pdv_all, kcal_all)

thresholds, check = {}, []
for ph in PHASES:
    score, gate, _, _ = res[ph]
    eligible = gate & quality_ok & plaus
    thr = float(np.quantile(score[eligible], 1 - params.loc[ph, "top_share"]))
    thresholds[ph] = thr
    label = eligible & (score >= thr)
    check.append({"phase": ph, "threshold": round(thr, 4),
                  "max_abs_score_diff": float(np.abs(score - recipes[f"score_{ph}"].to_numpy(float)).max()),
                  "label_mismatches": int((label != recipes[f"label_{ph}"].to_numpy(bool)).sum()),
                  "labelled": int(label.sum())})
check = pd.DataFrame(check).set_index("phase")
print(check)
assert (check["max_abs_score_diff"] < 1e-6).all() and (check["label_mismatches"] == 0).all(), \
    "phase function does not reproduce notebook 05: check phase_rules.csv / phase_params.csv / settings"
print("✓ phase function reproduces notebook 05")

            threshold  max_abs_score_diff  label_mismatches  labelled
phase                                                                
menstrual      0.3525                 0.0                 0      1952
follicular     0.3173                 0.0                 0      1866
ovulatory      0.3173                 0.0                 0      1866
luteal         0.3085                 0.0                 0      1529
✓ phase function reproduces notebook 05


## 3. Recipe details from the raw file

Food.com stores lists as R vectors, e.g. `c("2", "1/2")`. We parse them into Python lists. Quantities in Food.com have **no units**, so the ingredient lines read like "2 · all-purpose flour" (a known limitation of the dataset).

In [4]:
raw_cols = ["RecipeId", "Description", "Images", "Keywords", "RecipeIngredientQuantities",
            "RecipeIngredientParts", "RecipeInstructions", "RecipeServings", "RecipeYield"]
raw = pd.read_csv(RAW_RECIPES_CSV, usecols=lambda c: c in raw_cols, low_memory=False)
raw["RecipeId"] = pd.to_numeric(raw["RecipeId"], errors="coerce")
raw = raw.dropna(subset=["RecipeId"])
raw["RecipeId"] = raw["RecipeId"].astype("int64")
raw = raw[raw["RecipeId"].isin(set(catalog_ids))].drop_duplicates("RecipeId").set_index("RecipeId")
print(f"raw rows for catalog: {len(raw):,} of {n_cat:,}")

_STR = re.compile(r'"((?:[^"\\]|\\.)*)"')


def r_list(x):
    """Parse an R vector string like c("a", "b") into ['a', 'b']."""
    if not isinstance(x, str) or x.strip() in ("", "NA", "character(0)"):
        return []
    found = _STR.findall(x)
    if found:
        return [html.unescape(s.replace('\\"', '"')).strip() for s in found]
    return [html.unescape(x.strip())]


details = pd.DataFrame(index=catalog_ids)
details.index.name = "RecipeId"
details["description"] = raw["Description"].reindex(catalog_ids).fillna("").astype(str).map(html.unescape)
details["image_url"] = raw["Images"].reindex(catalog_ids).map(lambda x: (r_list(x) or [""])[0])
details["keywords"] = raw["Keywords"].reindex(catalog_ids).map(r_list)
qs = raw["RecipeIngredientQuantities"].reindex(catalog_ids).map(r_list)
ps = raw["RecipeIngredientParts"].reindex(catalog_ids).map(r_list)
details["ingredients_display"] = [
    [f"{q} · {p}".strip(" ·") if q and q != "NA" else p for q, p in zip(qq, pp)] if len(qq) == len(pp) else pp
    for qq, pp in zip(qs, ps)]
details["instructions"] = raw["RecipeInstructions"].reindex(catalog_ids).map(r_list)
details["yield"] = raw["RecipeYield"].reindex(catalog_ids).fillna("").astype(str)

print(f"with image: {(details['image_url'] != '').mean()*100:.1f}% | with instructions: {(details['instructions'].map(len) > 0).mean()*100:.1f}%")
details.head(3)

raw rows for catalog: 12,375 of 12,375
with image: 35.2% | with instructions: 100.0%


,description,image_url,keywords,ingredients_display,instructions,yield
RecipeId,,,,,,
55,Make and share this Betty Crocker's Southwestern Guacamole Dip rec...,"https://img.sndimg.com/food/image/upload/w_555,h_416,c_fit,fl_prog...","[Mexican, Low Protein, Low Cholesterol, Healthy, Weeknight, No Coo...","[5 · avocados, 1 · tomatoes, 1/2 · salt, 4 · garlic, 1/4 · lime ju...","[Peel and mash avocados until lumpy., Add remaining ingredients an...",
56,This recipe was originally noted by my wife on a cooking show on T...,"https://img.sndimg.com/food/image/upload/w_555,h_416,c_fit,fl_prog...","[Dessert, Oven, < 4 Hours, Easy]","[1/2 · butter, 1 1/2 · margarine, 3 · sugar, 3 · flour, 1 · eggs, ...","[Preheat oven to 400°F., Beat the butter and sugar together until ...",
85,Make and share this Butter Madeira Cake recipe from Food.com.,"https://img.sndimg.com/food/image/upload/w_555,h_416,c_fit,fl_prog...","[Weeknight, Oven, < 4 Hours]","[225 · flour, 2 · baking powder, 175 · caster sugar, 175 · butter,...","[Grease lined tin with melted butter., Sift flour and baking powde...",


## 4. Meal type

`meal_type_map.csv` maps each RecipeCategory to a meal type. The first run writes a **draft** from keyword rules; after that the file is yours (edit the `meal_type` column and re-run, it is never overwritten).

- `AUTO` = the category says nothing about the meal (e.g. "< 60 Mins", "Easy"): the type comes from the recipe's Keywords, and if they say nothing either, from the recipe **name** (e.g. "Guacamole Dip" → Snacks), using the same rules.
- Anything still unresolved becomes `Other` (shown only when the user picks "All").

In [5]:
MEAL_TYPES = ["Main", "Breakfast", "Salads & sides", "Snacks & appetizers", "Desserts & baking", "Sauces & dressings", "Other"]
RULES_MEAL = [   # (substring in lower-case text, meal type) — first match wins
    ("savory pie", "Main"), ("pot pie", "Main"),
    ("sauce", "Sauces & dressings"), ("dressing", "Sauces & dressings"), ("spread", "Sauces & dressings"),
    ("chutney", "Sauces & dressings"), ("jelly", "Sauces & dressings"), ("jellies", "Sauces & dressings"),
    ("marinade", "Sauces & dressings"), ("stock", "Sauces & dressings"), ("salsa", "Sauces & dressings"),
    ("jams", "Sauces & dressings"), ("condiment", "Sauces & dressings"), ("gravy", "Sauces & dressings"),
    ("breakfast", "Breakfast"), ("brunch", "Breakfast"), ("oatmeal", "Breakfast"), ("pancake", "Breakfast"),
    ("waffle", "Breakfast"), ("smoothie", "Breakfast"), ("shake", "Breakfast"), ("granola", "Breakfast"),
    ("cheesecake", "Desserts & baking"), ("dessert", "Desserts & baking"), ("cookie", "Desserts & baking"),
    ("cake", "Desserts & baking"), ("candy", "Desserts & baking"), ("frozen", "Desserts & baking"),
    ("ice cream", "Desserts & baking"), ("gelatin", "Desserts & baking"), ("tart", "Desserts & baking"),
    ("scone", "Desserts & baking"), ("muffin", "Desserts & baking"), ("bread", "Desserts & baking"),
    ("brownie", "Desserts & baking"), ("pudding", "Desserts & baking"), ("pie", "Desserts & baking"),
    ("frittata", "Breakfast"), ("omelet", "Breakfast"), ("crepe", "Breakfast"), ("french toast", "Breakfast"),
    ("fried rice", "Main"), ("chili", "Main"), ("casserole", "Main"), ("pizza", "Main"), ("taco", "Main"),
    ("burrito", "Main"), ("enchilada", "Main"), ("quiche", "Main"), ("burger", "Main"), ("sandwich", "Main"),
    ("wrap", "Main"), ("noodle", "Main"), ("ramen", "Main"), ("stir fry", "Main"), ("stroganoff", "Main"),
    ("risotto", "Main"), ("paella", "Main"), ("kebab", "Main"), ("fajita", "Main"), ("bake", "Main"),
    ("guacamole", "Snacks & appetizers"), ("hummus", "Snacks & appetizers"), ("bruschetta", "Snacks & appetizers"),
    ("slaw", "Salads & sides"), ("fritter", "Salads & sides"), ("mash", "Salads & sides"),
    ("soup", "Main"), ("stew", "Main"), ("chowder", "Main"), ("gumbo", "Main"), ("curr", "Main"),
    ("one dish", "Main"), ("chicken", "Main"), ("meat", "Main"), ("pork", "Main"), ("beef", "Main"),
    ("lamb", "Main"), ("veal", "Main"), ("ham", "Main"), ("steak", "Main"), ("poultry", "Main"),
    ("turkey", "Main"), ("duck", "Main"), ("fish", "Main"), ("salmon", "Main"), ("tuna", "Main"),
    ("tilapia", "Main"), ("trout", "Main"), ("catfish", "Main"), ("halibut", "Main"), ("crab", "Main"),
    ("lobster", "Main"), ("shellfish", "Main"), ("shrimp", "Main"), ("seafood", "Main"), ("meatball", "Main"),
    ("spaghetti", "Main"), ("penne", "Main"), ("pasta", "Main"), ("lasagna", "Main"), ("macaroni", "Main"),
    ("tofu", "Main"), ("soy", "Main"), ("lentil", "Main"), ("roast", "Main"), ("game", "Main"),
    ("deer", "Main"), ("rabbit", "Main"), ("main", "Main"), ("dinner", "Main"),
    ("lunch", "Snacks & appetizers"), ("snack", "Snacks & appetizers"), ("appetizer", "Snacks & appetizers"),
    ("cheese", "Snacks & appetizers"), ("dip", "Snacks & appetizers"), ("nuts", "Snacks & appetizers"),
    ("vegetable", "Salads & sides"), ("potato", "Salads & sides"), ("rice", "Salads & sides"),
    ("grain", "Salads & sides"), ("bean", "Salads & sides"), ("green", "Salads & sides"), ("corn", "Salads & sides"),
    ("salad", "Salads & sides"), ("yam", "Salads & sides"), ("squash", "Salads & sides"), ("onion", "Salads & sides"),
    ("pepper", "Salads & sides"), ("cauliflower", "Salads & sides"), ("spinach", "Salads & sides"),
    ("artichoke", "Salads & sides"), ("broccoli", "Salads & sides"), ("carrot", "Salads & sides"),
    ("mushroom", "Salads & sides"), ("cabbage", "Salads & sides"), ("asparagus", "Salads & sides"),
    ("eggplant", "Salads & sides"), ("tomato", "Salads & sides"), ("beet", "Salads & sides"),
    ("chard", "Salads & sides"), ("collard", "Salads & sides"), ("okra", "Salads & sides"),
    ("peas", "Salads & sides"), ("radish", "Salads & sides"), ("leek", "Salads & sides"),
    ("zucchini", "Salads & sides"), ("side", "Salads & sides"),
    ("beverage", "Other"), ("punch", "Other"), ("drink", "Other"), ("cocktail", "Other"), ("coffee", "Other"),
]


_RULE_RX = [(re.compile(r"\b" + re.escape(key)), mt) for key, mt in RULES_MEAL]   # match at word starts only ("bean" not in "Caribbean")


def rule_type(text):
    t = str(text).lower()
    for rx, mt in _RULE_RX:
        if rx.search(t):
            return mt
    return None


cat_counts = recipes.set_index("RecipeId").loc[catalog_ids, "RecipeCategory"].fillna("(missing)").value_counts()
if not os.path.exists(MEAL_MAP_CSV):
    draft = pd.DataFrame({"RecipeCategory": cat_counts.index, "n_catalog": cat_counts.to_numpy()})
    draft["meal_type"] = draft["RecipeCategory"].map(lambda c: rule_type(c) or "AUTO")
    draft.to_csv(MEAL_MAP_CSV, index=False, encoding="utf-8-sig")
    print(f"wrote draft {MEAL_MAP_CSV}: review the meal_type column, then re-run this cell")
meal_map = pd.read_csv(MEAL_MAP_CSV)
bad = set(meal_map["meal_type"]) - set(MEAL_TYPES) - {"AUTO"}
assert not bad, f"unknown meal types in {MEAL_MAP_CSV}: {bad}"
meal_map = meal_map.set_index("RecipeCategory")["meal_type"]

cats = recipes.set_index("RecipeId").loc[catalog_ids, "RecipeCategory"].fillna("(missing)")
meal = cats.map(meal_map).fillna("AUTO")
auto = meal == "AUTO"
auto_ids = auto[auto].index
from_kw = details.loc[auto_ids, "keywords"].map(lambda kws: next((rule_type(k) for k in kws if rule_type(k)), None))
from_name = rec_by_id.loc[auto_ids, "Name"].map(rule_type)          # 3rd step: the recipe name
resolved = from_kw.fillna(from_name)
meal.loc[auto] = resolved.fillna("Other").to_numpy()
details["meal_type"] = meal.to_numpy()

print(f"categories mapped directly: {(meal_map != 'AUTO').sum()} | AUTO: {(meal_map == 'AUTO').sum()}")
print(f"AUTO recipes: {int(auto.sum()):,} | resolved from keywords: {int(from_kw.notna().sum()):,} | from name: {int((from_kw.isna() & from_name.notna()).sum()):,} | still Other: {int(resolved.isna().sum()):,}")
summary_meal = details["meal_type"].value_counts().rename("n_recipes").to_frame()
summary_meal["%"] = (summary_meal["n_recipes"] / n_cat * 100).round(1)
summary_meal

categories mapped directly: 84 | AUTO: 93
AUTO recipes: 1,459 | resolved from keywords: 105 | from name: 1,070 | still Other: 284


,n_recipes,%
meal_type,,
Desserts & baking,6319,51.1
Salads & sides,1856,15.0
Main,1797,14.5
Breakfast,794,6.4
Snacks & appetizers,712,5.8
Other,477,3.9
Sauces & dressings,420,3.4


In [6]:
# review aid: biggest AUTO categories and a sample of each meal type
print("Largest AUTO categories (resolved per recipe from keywords):")
print(meal_map[meal_map == "AUTO"].to_frame().join(cat_counts.rename("n_catalog")).sort_values("n_catalog", ascending=False).head(15).to_string())
for mt in MEAL_TYPES:
    ids = details.index[details["meal_type"] == mt][:5]
    print(f"\n{mt}: " + " | ".join(rec_by_id.loc[ids, "Name"].tolist()))

Largest AUTO categories (resolved per recipe from keywords):
                meal_type  n_catalog
RecipeCategory                      
Low Protein          AUTO        232
< 30 Mins            AUTO        152
< 60 Mins            AUTO        118
Low Cholesterol      AUTO        105
< 15 Mins            AUTO         71
Very Low Carbs       AUTO         69
Healthy              AUTO         60
European             AUTO         57
< 4 Hours            AUTO         55
Fruit                AUTO         53
Apple                AUTO         43
High In...           AUTO         36
High Protein         AUTO         28
Mexican              AUTO         27
Kid Friendly         AUTO         22

Main: White Chili | Ramen Foo Yung | Chicken with Cheddar and Green Chilies | Solo Chicken Breast and Bell Pepper | Chicken Curry III

Breakfast: Basic Crepes II | Butter Crepes | Tjock Pan Kaka (Swedish Thick Pancake) | Southwest Smoothie | Banana and Kiwi Smoothie

Salads & sides: Sun-Dried Tomato Pesto | 

## 5. Popularity (all interactions)

Same weighted rating as notebook 06 (Bayesian average, m = 75th percentile of rating counts), now on **all** interactions. Saved as a 0–1 rank per catalog recipe (1 = most popular).

In [7]:
g = inter.groupby("recipe_id")
pst = pd.DataFrame({"n_interactions": g.size(), "n_ratings": g["has_rating"].sum(), "mean_rating": g["rating"].mean()})
C_mean = inter["rating"].mean()
M_min = float(pst.loc[pst["n_ratings"] > 0, "n_ratings"].quantile(0.75))
pst["weighted_rating"] = (pst["n_ratings"] * pst["mean_rating"].fillna(C_mean) + M_min * C_mean) / (pst["n_ratings"] + M_min)
pst = pst.reindex(catalog_ids)
order = np.lexsort((pst["n_interactions"].fillna(0).to_numpy(), pst["weighted_rating"].fillna(-1).to_numpy()))
pop_rank = np.empty(n_cat, dtype=np.float32)
pop_rank[order] = np.arange(n_cat) / (n_cat - 1)
print(f"global mean = {C_mean:.3f} | m = {M_min:.0f} | catalog recipes with interactions: {pst['n_interactions'].notna().sum():,}")
top = catalog_ids[np.argsort(-pop_rank)[:8]]
rec_by_id.loc[top, ["Name"]].join(details[["meal_type"]])

global mean = 4.657 | m = 5 | catalog recipes with interactions: 6,960


,Name,meal_type
RecipeId,,
52285,Cream Cheese Flan,Desserts & baking
141488,Snickerdoodles,Desserts & baking
244193,Absolutely the Best New York Cheesecake (Gluten-Free),Desserts & baking
404470,Fluffy Italian Bread,Desserts & baking
102451,Whipped Cauliflower Dijon,Salads & sides
68670,Italian Style Skillet Zucchini,Salads & sides
254570,Shirley's Plain or Blueberry Muffins,Breakfast
113193,"Sweet, Buttery Rolls (Bread Machine Recipe)",Desserts & baking


## 6. Content-Based: TF-IDF

Fitted on all 42,736 recipes (same as notebook 07), saved for the catalog only. The vocabulary and IDF values are saved too, so the app can build a TF-IDF row for the user's **own** recipes in exactly the same way.

In [8]:
vec = TfidfVectorizer(analyzer=lambda parts: list(parts), lowercase=False)
X = vec.fit_transform(recipes["parts_norm"]).tocsr()
rid_to_row = pd.Series(np.arange(len(recipes)), index=recipes["RecipeId"])
X_cat = X[rid_to_row.loc[catalog_ids].to_numpy()].astype(np.float32)
vocab = vec.get_feature_names_out().tolist()
sparse.save_npz(out_path("tfidf_catalog.npz"), X_cat)
with open(out_path("tfidf_vocab.json"), "w", encoding="utf-8") as f:
    json.dump({"vocab": vocab, "idf": vec.idf_.round(6).tolist()}, f)


def tfidf_row(parts):
    """TF-IDF row for a list of ingredients (the app re-implements this for own recipes)."""
    idx = {w: i for i, w in enumerate(vocab)}
    v = np.zeros(len(vocab))
    for p in parts:
        if p in idx:
            v[idx[p]] += 1
    v *= vec.idf_
    n = np.linalg.norm(v)
    return v / n if n > 0 else v


i0 = 0
assert np.allclose(tfidf_row(recipes.loc[rid_to_row[catalog_ids[i0]], "parts_norm"]), X_cat[i0].toarray().ravel(), atol=1e-5)
print(f"TF-IDF catalog: {X_cat.shape} | vocab {len(vocab)} | ✓ own-recipe row matches sklearn")

TF-IDF catalog: (12375, 481) | vocab 481 | ✓ own-recipe row matches sklearn


## 7. Collaborative Filtering (all interactions)

- **Item-CF:** Jaccard between catalog recipes from who interacted with them; keep the top `ITEM_K` = 10 neighbours per recipe. The user's hearts are always catalog recipes, so catalog × catalog is enough.
- **User-CF:** the app compares the user's hearts with every Food.com user at request time. It needs: each user's interactions with catalog recipes, their likes (4+) on catalog recipes, and their **total** number of interactions (the Jaccard denominator).

In [9]:
all_users = np.sort(inter["user_id"].unique())
uid_row = pd.Series(np.arange(len(all_users)), index=all_users)
n_users = len(all_users)
ic = inter[inter["recipe_id"].isin(cat_index.index)]
rows, cols = uid_row.loc[ic["user_id"]].to_numpy(), cat_index.loc[ic["recipe_id"]].to_numpy()
B_cat = sparse.csr_matrix((np.ones(len(ic), np.float32), (rows, cols)), shape=(n_users, n_cat))
lk = ic["liked"].to_numpy(bool)
L_cat = sparse.csr_matrix((np.ones(lk.sum(), np.float32), (rows[lk], cols[lk])), shape=(n_users, n_cat))
user_total = inter.groupby("user_id").size().reindex(all_users).to_numpy(np.float32)

# Item-CF: Jaccard on catalog columns
cnt = np.asarray(B_cat.sum(axis=0)).ravel()
Bt = B_cat.T.tocsr()
r_l, c_l, v_l = [], [], []
for s in range(0, n_cat, CHUNK):
    cols_ = np.arange(s, min(s + CHUNK, n_cat))
    co = (Bt @ B_cat[:, cols_]).toarray()
    denom = cnt[:, None] + cnt[cols_][None, :] - co
    sim = np.divide(co, denom, out=np.zeros_like(co), where=denom > 0)
    sim[cols_, np.arange(len(cols_))] = 0
    top = np.argpartition(-sim, ITEM_K, axis=0)[:ITEM_K]
    tv = np.take_along_axis(sim, top, axis=0)
    r_l.append(top.T.ravel()); v_l.append(tv.T.ravel()); c_l.append(np.repeat(cols_, ITEM_K))
r_, c_, v_ = np.concatenate(r_l), np.concatenate(c_l), np.concatenate(v_l)
keep = v_ > 0
S_item = sparse.csr_matrix((v_[keep].astype(np.float32), (r_[keep], c_[keep])), shape=(n_cat, n_cat))

sparse.save_npz(out_path("itemknn_catalog.npz"), S_item)
sparse.save_npz(out_path("usercf_interactions.npz"), B_cat)
sparse.save_npz(out_path("usercf_liked.npz"), L_cat)
np.save(out_path("usercf_counts.npy"), user_total)
print(f"users: {n_users:,} | B_cat nnz {B_cat.nnz:,} | L_cat nnz {L_cat.nnz:,} | item neighbours nnz {S_item.nnz:,}")
print(f"catalog recipes with no neighbour: {int((np.asarray((S_item > 0).sum(axis=0)).ravel() == 0).sum()):,}")

users: 60,183 | B_cat nnz 50,158 | L_cat nnz 43,993 | item neighbours nnz 53,234
catalog recipes with no neighbour: 6,324


## 8. Ingredient table for own recipes

For each ingredient in the list: USDA nutrients per 100 g and grams per unit (cup, tbsp, tsp, piece, oz), built the same way as notebook 04. `g` (grams) is always available.

In [10]:
imap = pd.read_csv(MAP_CSV)
per100 = pd.read_pickle(PER100_PKL)
portions = pd.read_csv(PORTIONS_CSV)
du = pd.read_csv(DEFAULT_UNITS_CSV)[["ingredient", "unit"]].rename(columns={"unit": "default_unit"})

NUT_COLS = ["kcal"] + [k for k in DV if k in per100.columns]
ing = imap[["ingredient", "fdc_id", "usda_description"]].dropna(subset=["fdc_id"]).drop_duplicates("ingredient").copy()
ing["fdc_id"] = ing["fdc_id"].astype(int)
ing = ing.merge(per100[NUT_COLS].reset_index(), on="fdc_id", how="left").dropna(subset=["kcal"])

p = portions.dropna(subset=["gram_weight"]).copy()
p["amount"] = pd.to_numeric(p["amount"], errors="coerce").fillna(1).replace(0, 1)
p["gpu"] = p["gram_weight"] / p["amount"]
p["label"] = p["portion_label"].fillna("").astype(str).str.lower().str.strip()
SKIP = ("serving", "nlea", "package", "container", "packet", "can ", "bottle", "jar", "cubic", "pat ", "dash", "drop", "scoop")


def unit_class(label):
    if label.startswith("tsp"):
        return "tsp"
    if label.startswith("tbsp"):
        return "tbsp"
    if label.startswith("cup"):
        return "cup"
    if label.startswith("fl oz"):
        return "floz"
    if label == "oz" or label.startswith("oz "):
        return "oz"
    if label == "" or any(s in label for s in SKIP):
        return None
    return "piece"


p["uc"] = p["label"].apply(unit_class)
p = p.dropna(subset=["uc"]).copy()
p["exact"] = -(p["label"] == p["uc"]).astype(int)
first = p.sort_values(["ingredient", "uc", "exact", "seq_num"]).groupby(["ingredient", "uc"]).first().reset_index()
U = first.pivot(index="ingredient", columns="uc", values="gpu")
for c in ["cup", "tbsp", "tsp", "floz", "oz", "piece"]:
    if c not in U.columns:
        U[c] = np.nan
U = U.reindex(ing["ingredient"])
U["cup"] = U["cup"].fillna(U["tbsp"] * 16).fillna(U["tsp"] * 48).fillna(U["floz"] * 8)
U["tbsp"] = U["tbsp"].fillna(U["cup"] / 16)
U["tsp"] = U["tsp"].fillna(U["tbsp"] / 3)
U["oz"] = U["oz"].fillna(28.3495)
piece_label = first[first["uc"] == "piece"].set_index("ingredient")["label"]

ing_app = ing.set_index("ingredient")
for u in ["cup", "tbsp", "tsp", "piece", "oz"]:
    ing_app[f"g_per_{u}"] = U[u].round(2)
ing_app["piece_label"] = piece_label.reindex(ing_app.index).fillna("")
ing_app = ing_app.join(du.set_index("ingredient"), how="left")
ing_app["n_recipes"] = pd.Series([x for parts in recipes["parts_norm"] for x in parts]).value_counts().reindex(ing_app.index).fillna(0).astype(int)
ing_app = ing_app.sort_values("n_recipes", ascending=False)
ing_app.reset_index().to_csv(out_path("ingredients_app.csv"), index=False, encoding="utf-8-sig")
print(f"ingredients for own recipes: {len(ing_app)} | in TF-IDF vocab: {ing_app.index.isin(vocab).mean()*100:.0f}%")
ing_app[["usda_description", "kcal", "iron_mg", "g_per_cup", "g_per_tbsp", "g_per_piece", "piece_label", "default_unit"]].head(10)

ingredients for own recipes: 489 | in TF-IDF vocab: 98%


,usda_description,kcal,iron_mg,g_per_cup,g_per_tbsp,g_per_piece,piece_label,default_unit
ingredient,,,,,,,,
salt,"Salt, table",0.0,0.33,292.0,18.00,NaN,,tsp
butter,"Butter, salted",717.0,0.02,227.0,14.20,113.0,stick,tbsp
sugar,"Sugars, granulated",387.0,0.05,200.0,12.50,NaN,,cup
all purpose flour,"Wheat flour, white, all-purpose, enriched, bleached",364.0,4.64,125.0,7.81,NaN,,cup
egg,"Egg, whole, raw, fresh",143.0,1.75,243.0,15.19,50.0,large,cup
vanilla extract,Vanilla extract,288.0,0.12,208.0,13.00,NaN,,tsp
water,"Beverages, water, tap, drinking",0.0,0.00,236.8,14.80,1000.0,liter,cup
baking powder,"Leavening agents, baking powder, double-acting, sodium aluminum su...",53.0,11.02,220.8,13.80,NaN,,tsp
milk,"Milk, whole, 3.25% milkfat, with added vitamin D",61.0,0.03,244.0,15.00,976.0,quart,cup


## 9. Pantry staples and quick-start recipes

- `pantry_staples.csv` (manual, created once): ingredients counted as always at home. Edit it freely.
- `onboarding_ids.csv`: about 12 popular recipes across meal types for the "pick 3 you like" start screen.

In [11]:
STAPLES_DEFAULT = ["salt", "black pepper", "pepper", "water", "vegetable oil", "olive oil", "oil", "sugar",
                   "all purpose flour", "flour", "baking powder", "baking soda", "butter", "garlic"]
if not os.path.exists(STAPLES_CSV):
    pd.DataFrame({"ingredient": [s for s in STAPLES_DEFAULT if s in set(vocab)]}).to_csv(STAPLES_CSV, index=False, encoding="utf-8-sig")
    print(f"wrote draft {STAPLES_CSV}")
staples = pd.read_csv(STAPLES_CSV)["ingredient"].tolist()
missing = [s for s in staples if s not in set(vocab)]
print("staples:", staples)
if missing:
    print("⚠ not in the ingredient vocabulary (ignored by the app):", missing)
pd.DataFrame({"ingredient": staples}).to_csv(out_path("pantry_staples.csv"), index=False, encoding="utf-8-sig")

ONBOARD = {"Main": 4, "Breakfast": 2, "Salads & sides": 2, "Snacks & appetizers": 2, "Desserts & baking": 2}
pick = []
for mt, n in ONBOARD.items():
    idx = np.where(details["meal_type"].to_numpy() == mt)[0]
    pick += list(catalog_ids[idx[np.argsort(-pop_rank[idx])[:n]]])
onboard = pd.DataFrame({"RecipeId": pick})
onboard.to_csv(out_path("onboarding_ids.csv"), index=False, encoding="utf-8-sig")
onboard.join(rec_by_id[["Name"]], on="RecipeId").join(details[["meal_type"]], on="RecipeId")

staples: ['salt', 'black pepper', 'water', 'olive oil', 'sugar', 'all purpose flour', 'baking powder', 'baking soda', 'butter', 'garlic']


,RecipeId,Name,meal_type
0,352692,Easy Crispy Dijon Chicken,Main
1,219572,Creamy Macaroni & Cheese - for Two or One,Main
2,71095,Mediterranean Fish,Main
3,183217,Chicken Georgia,Main
4,254570,Shirley's Plain or Blueberry Muffins,Breakfast
5,425703,Greek Yogurt With Honey and Walnuts,Breakfast
6,102451,Whipped Cauliflower Dijon,Salads & sides
7,68670,Italian Style Skillet Zucchini,Salads & sides
8,26006,3-Ingredient Fruit Dip,Snacks & appetizers
9,100407,Spinach and Artichoke Dip,Snacks & appetizers


## 10. Recipe table for the app

In [12]:
base = rec_by_id.loc[catalog_ids, ["Name", "RecipeCategory", "TotalTime_min", "Calories", "servings_final", "parts_norm"]
                     + [f"{c}_{ph}" for ph in PHASES for c in ("score", "label", "why")]].copy()
pdv_cat = pdv_all.set_index(nut["RecipeId"]).loc[catalog_ids].round(1)
pdv_cat.columns = [f"pdv_{k}" for k in pdv_cat.columns]
app = base.join(pdv_cat).join(details[["meal_type", "description", "image_url", "ingredients_display", "instructions", "yield"]])
app["parts_norm"] = app["parts_norm"].map(list)
app = app.reset_index()
assert (app["RecipeId"].to_numpy() == catalog_ids).all()

try:
    app.to_parquet(out_path("recipes_app.parquet"), index=False)
except ImportError:
    raise ImportError("parquet needs pyarrow: %pip install pyarrow (and add it to requirements.txt)")
print(app.shape)
app.drop(columns=["description", "instructions", "ingredients_display"]).head(3)

(12375, 36)


,RecipeId,Name,RecipeCategory,TotalTime_min,Calories,servings_final,parts_norm,score_menstrual,label_menstrual,why_menstrual,...,pdv_zinc_mg,pdv_vitb6_mg,pdv_folate_ug,pdv_vitb12_ug,pdv_fiber_g,pdv_protein_g,pdv_potassium_mg,meal_type,image_url,yield
0,55,Betty Crocker's Southwestern Guacamole Dip,Southwestern U.S.,125,415.9,4.0,"[avocado, tomato, salt, garlic, lime juice]",0.266373,False,,...,17.3,63.0,48.7,0.0,55.4,13.3,27.4,Snacks & appetizers,"https://img.sndimg.com/food/image/upload/w_555,h_416,c_fit,fl_prog...",
1,56,Buttermilk Pie,Pie,80,395.9,8.0,"[butter, margarine, sugar, all purpose flour, egg, salt, vanilla e...",0.095435,False,,...,5.5,3.4,17.9,10.1,3.2,13.7,2.1,Desserts & baking,"https://img.sndimg.com/food/image/upload/w_555,h_416,c_fit,fl_prog...",
2,85,Butter Madeira Cake,Dessert,175,373.8,8.0,"[all purpose flour, baking powder, sugar, butter, egg, milk, vanil...",0.140015,False,,...,10.4,8.0,18.2,31.0,2.1,23.4,3.3,Desserts & baking,"https://img.sndimg.com/food/image/upload/w_555,h_416,c_fit,fl_prog...",


## 11. Phase files, config and sizes

In [13]:
rules.to_csv(out_path("phase_rules.csv"), index=False, encoding="utf-8-sig")
params.reset_index().to_csv(out_path("phase_params.csv"), index=False, encoding="utf-8-sig")
pd.Series(thresholds, name="threshold").rename_axis("phase").reset_index().to_csv(out_path("phase_thresholds.csv"), index=False, encoding="utf-8-sig")
pd.DataFrame([{"nutrient": k, "dv": v[0], "unit": v[1], "name": v[2]} for k, v in DV.items()]).to_csv(out_path("daily_values.csv"), index=False, encoding="utf-8-sig")

config = {
    "weights": WEIGHTS, "item_k": ITEM_K, "user_k": USER_K, "like_threshold": LIKE_THRESHOLD,
    "max_missing_default": MAX_MISSING_DEFAULT, "own_top10_cap": OWN_TOP10_CAP, "own_unknown_max": OWN_UNKNOWN_MAX,
    "meal_types": MEAL_TYPES, "meal_types_hidden_by_default": ["Sauces & dressings"], "meal_type_default": ["Main", "Salads & sides"],
    "phases": PHASES,
    "phase_scoring": {"max_pct_dv": MAX_PCT_DV, "min_kcal_eligible": MIN_KCAL_ELIGIBLE, "max_kcal": MAX_KCAL,
                      "density_cap": DENSITY_CAP, "min_kcal_density": MIN_KCAL_DENSITY},
    "n_catalog": int(n_cat), "n_users": int(n_users),
}
with open(out_path("app_config.json"), "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2)

sizes = pd.DataFrame([(f, os.path.getsize(out_path(f)) / 1e6) for f in sorted(os.listdir(OUT_DIR))], columns=["file", "MB"]).round(2)
print(sizes.to_string(index=False))
print(f"\ntotal: {sizes['MB'].sum():.1f} MB (GitHub limit per file: 100 MB)")

                   file   MB
        app_config.json 0.00
       daily_values.csv 0.00
    ingredients_app.csv 0.07
    itemknn_catalog.npz 0.16
     onboarding_ids.csv 0.00
     pantry_staples.csv 0.00
       phase_params.csv 0.00
        phase_rules.csv 0.00
   phase_thresholds.csv 0.00
    recipes_app.parquet 6.46
      tfidf_catalog.npz 0.47
       tfidf_vocab.json 0.01
      usercf_counts.npy 0.24
usercf_interactions.npz 0.14
       usercf_liked.npz 0.12

total: 7.7 MB (GitHub limit per file: 100 MB)
